In [1]:
import json
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew credits for a movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of similar movies for a movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {"type": "integer", "description": "The movie ID"}
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [3]:
def call_tool(name, args):
    if name == "get_popular_movies":
        url = f"{BASE_URL}/movies"
    elif name == "get_movie_details":
        url = f"{BASE_URL}/movies/{args['movie_id']}"
    elif name == "get_movie_credits":
        url = f"{BASE_URL}/movies/{args['movie_id']}/credits"
    elif name == "get_similar_movies":
        url = f"{BASE_URL}/movies/{args['movie_id']}/similar"
    else:
        return json.dumps({"error": f"Unknown tool: {name}"})
    return requests.get(url).text

In [4]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a personalized movie recommendation chatbot.\n"
            "Your job is to recommend movies based on the user's taste.\n\n"
            "Rules:\n"
            "- When the user mentions a genre or preference, immediately look up movies and recommend them.\n"
            "- Do NOT ask follow-up questions before recommending. Act on what the user gives you right away.\n"
            "- Remember the user's favorite genres from the conversation.\n"
            "- Remember movies the user has already watched.\n"
            "- NEVER recommend a movie the user said they already watched.\n"
            "- Use the provided tools to look up real movie data before answering.\n"
            "- When recommending, explain briefly why it matches the user's taste.\n"
            "- Respond in the same language the user uses."
        ),
    },
]


def chat():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"[Tool Call] {tc.function.name}({args})")
                result = call_tool(tc.function.name, args)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result,
                    }
                )
        else:
            messages.append({"role": "assistant", "content": msg.content})
            print(f"AI: {msg.content}")
            return

In [5]:
while True:
    message = input("You: ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        chat()

User: 저는 SF 영화를 좋아해요
[Tool Call] get_popular_movies({})
AI: 여기 SF 장르의 인기 있는 영화 몇 편을 추천해 드립니다:

1. **Mercy (2026-01-20)**  
   ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)  
   가까운 미래에 한 형사가 아내를 살해한 혐의로 재판을 받으며, 그 자신의 무죄를 입증하기 위해 고급 AI 판사와 대결하는 이야기를 그립니다. AI와의 갈등과 인간성에 대한 깊은 성찰이 돋보입니다.

2. **28 Years Later: The Bone Temple (2026-01-14)**  
   ![28 Years Later: The Bone Temple](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)  
   공포와 스릴을 결합한 이 영화는, 주인공이 어떠한 비극적인 사건에 휘말리며 펼쳐지는 이야기를 담고 있습니다. SF 요소가 추가되어 긴장감이 넘치는 플롯이 특징입니다.

3. **Greenland 2: Migration (2026-01-07)**  
   ![Greenland 2: Migration](https://image.tmdb.org/t/p/w780/z2tqCJLsw6uEJ8nJV8BsQXGa3dr.jpg)  
   인류의 재난을 배경으로 한 이 영화는 가족이 새로운 안식을 찾기 위해 위험한 여정을 떠나는 이야기를 다룹니다. 안전과 생존을 위한 눈물겨운 싸움이 그려집니다.

4. **Space/Time (2025-05-01)**  
   ![Space/Time](https://image.tmdb.org/t/p/w780/uje1ecKMnNpZp0at5TxlvVgVXqI.jpg)  
   실패한 실험 후 범죄의 세계에 발을 들여놓은 과학자들이 인류를 구하기 위한 공간 왜곡 엔진을 재건하려는 이야기입니다. 과학과 범죄의 